# Gemini 03. 연결 확인과 에러 처리

Claude 05 노트북처럼 실패 상황을 **일부러 재현**해서 어떤 예외가 나는지 확인한다.

## 먼저 알아둘 것: Claude SDK와 다른 점 (google-genai 2.31.0에서 직접 확인)

| | Claude (`anthropic`) | Gemini (`google-genai`) |
|---|---|---|
| **기본 재시도** | 2번 자동 재시도 | **재시도 안 함** (`retry_options`를 지정해야 켜짐) |
| **기본 timeout** | 10분 | **없음** (무한정 기다림) |
| timeout 단위 | 초 | **밀리초** (`timeout=30_000` = 30초) |
| 서버 에러 응답 | 상태 코드별 클래스 (`AuthenticationError` 등) | `ClientError`(4xx) / `ServerError`(5xx) 두 가지. 세부는 `e.code`, `e.status`로 구분 |
| 연결 실패, 시간 초과 | SDK 예외 (`APIConnectionError` 등) | SDK 예외가 아니라 **`httpx` 라이브러리 예외**가 그대로 올라옴 |
| 잘못된 API Key | 401 | **400 `INVALID_ARGUMENT`** (reason: `API_KEY_INVALID`) |
| Rate Limit 확인 | 응답 헤더 (`anthropic-ratelimit-*`) | **헤더 없음** → AI Studio 화면에서 확인 |

> 공식 문서에는 "Python SDK가 일시적 오류를 최대 4번 재시도한다"고 되어 있지만, 이 버전에서 기본 클라이언트의 설정을 확인해 보면 `retry_options=None`, `timeout=None`이고 재시도하지 않는다. 그래서 아래에서는 **재시도와 timeout을 직접 설정**하는 방법을 함께 다룬다.

## 예외 구조

```
google.genai.errors.APIError        서버가 에러로 응답함 (e.code, e.status, e.message, e.details)
├── ClientError                     4xx: 요청이나 설정 문제
└── ServerError                     5xx: Google 서버 문제

httpx.TimeoutException              시간 초과 (ReadTimeout, ConnectTimeout 등)
httpx.ConnectError                  서버에 닿지 못함
```

## 에러별 정리 ([공식 에러 문서](https://ai.google.dev/gemini-api/docs/api-errors) 기준)

| 코드 | status | 흔한 원인 | 재시도 |
|---|---|---|---|
| 400 | `INVALID_ARGUMENT` | 파라미터 오류, **잘못된 API Key** | X |
| 400 | `FAILED_PRECONDITION` | 결제 미설정 등 전제조건 미충족 | X |
| 402 | - | **선불(Prepay) 크레딧 소진** | X (충전 전까지 실패) |
| 403 | `PERMISSION_DENIED` | 키 권한 없음, 차단된 키 | X |
| 404 | `NOT_FOUND` | 모델 이름 오타 | X |
| 429 | `RESOURCE_EXHAUSTED` | 분당/일일 한도 초과, 지출 한도 초과 | O (기다린 뒤) |
| 500 | `INTERNAL` | 서버 내부 오류 | O |
| 503 | `UNAVAILABLE` | 서버 과부하 | O |
| 504 | `DEADLINE_EXCEEDED` | 처리 시간 초과 | O |

In [1]:
import time

import httpx
from google import genai
from google.genai import errors, types
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

MODEL = "gemini-3.8-flash"


def error_reason(e: errors.APIError) -> str | None:
    """에러 상세 정보에서 reason(예: API_KEY_INVALID)을 꺼냄. 없으면 None"""
    try:
        for d in e.details["error"].get("details", []):
            if "reason" in d:
                return d["reason"]
    except (TypeError, KeyError, AttributeError):
        pass
    return None

## 1. API 연결 확인 (Health Check)

모델 목록 조회(`models.list`)는 토큰을 쓰지 않으므로 **무료**다. `page_size=1`로 1개만 받아 가볍게 확인한다.
확인용 클라이언트는 **10초 timeout, 재시도 없음**으로 따로 만든다.

In [2]:
check_client = genai.Client(
    http_options=types.HttpOptions(
        timeout=10_000,  # 밀리초 단위 → 10초
        retry_options=types.HttpRetryOptions(attempts=1),  # 1번만 시도 (재시도 없음)
    )
)


def check_connection(c: genai.Client) -> bool:
    start = time.perf_counter()
    try:
        c.models.list(config={"page_size": 1})
    except errors.ClientError as e:
        reason = error_reason(e)
        if reason == "API_KEY_INVALID":
            print(f"[실패] API Key가 잘못됨 ({e.code} {e.status})")
        else:
            print(f"[실패] 요청 오류 ({e.code} {e.status}): {e.message}")
    except errors.ServerError as e:
        print(f"[실패] Google 서버 오류 ({e.code} {e.status}): {e.message}")
    except httpx.TimeoutException:
        print("[실패] 시간 초과: 10초 안에 응답 없음")
    except httpx.ConnectError as e:
        print(f"[실패] 연결 오류: 서버에 닿지 못함 ({e})")
    else:
        print(f"[성공] 연결 정상 ({(time.perf_counter() - start) * 1000:.0f} ms)")
        return True
    return False


is_connected = check_connection(check_client)

[성공] 연결 정상 (363 ms)


## 2. 인증 오류: 잘못된 API Key

Claude는 401이었지만 **Gemini는 400 `INVALID_ARGUMENT`** 로 응답한다.
같은 400이 일반 파라미터 오류에도 쓰이므로, **키 문제인지는 `reason`(`API_KEY_INVALID`)으로 구분**해야 한다.

In [3]:
bad_client = genai.Client(api_key="invalid-key")

try:
    bad_client.models.list(config={"page_size": 1})
except errors.ClientError as e:
    print("예외 클래스 :", type(e).__name__)
    print("코드        :", e.code)
    print("status      :", e.status)
    print("reason      :", error_reason(e))
    print("메시지      :", e.message)

예외 클래스 : ClientError
코드        : 400
status      : INVALID_ARGUMENT
reason      : API_KEY_INVALID
메시지      : API key not valid. Please pass a valid API key.


## 3. Timeout (시간 초과)

### 3-1. timeout 설정 방법

```python
# 클라이언트 전체에 적용 (밀리초 단위!)
client = genai.Client(http_options=types.HttpOptions(timeout=60_000))

# 요청 하나에만 적용
client.models.generate_content(
    model=MODEL, contents="...",
    config=types.GenerateContentConfig(http_options=types.HttpOptions(timeout=60_000)),
)
```

### 3-2. 재현: 아주 짧은 timeout

0.05초(50ms)로 설정해 일부러 시간 초과를 낸다. 시간 초과는 SDK 예외가 아니라 **`httpx` 예외**로 올라온다.

In [4]:
fast_client = genai.Client(
    http_options=types.HttpOptions(timeout=50, retry_options=types.HttpRetryOptions(attempts=1))
)

try:
    fast_client.models.list(config={"page_size": 1})
except httpx.TimeoutException as e:
    print("예외 클래스 :", f"{type(e).__module__}.{type(e).__name__}")
    print("메시지      :", e)
except httpx.ConnectError as e:
    # timeout이 너무 짧으면 연결 단계에서 실패해 ConnectError로 나오는 경우도 있음
    print("예외 클래스 :", f"{type(e).__module__}.{type(e).__name__}  (연결 단계에서 시간 부족)")
    print("메시지      :", e)

예외 클래스 : httpx.ReadTimeout
메시지      : The read operation timed out


### 3-3. 주의: 생성 요청은 timeout을 10초 미만으로 둘 수 없다

`generate_content`에 10초보다 짧은 timeout을 주면, 시간 초과가 아니라 **서버가 바로 400으로 거절**한다.
(서버가 요청을 처리하지 않으므로 요금은 발생하지 않는다)

In [5]:
short_client = genai.Client(
    http_options=types.HttpOptions(timeout=2_000, retry_options=types.HttpRetryOptions(attempts=1))
)

try:
    short_client.models.generate_content(model=MODEL, contents="안녕")
except errors.ClientError as e:
    print(f"[{type(e).__name__} / {e.code} {e.status}] {e.message}")

[ClientError / 400 INVALID_ARGUMENT] Manually set deadline 2s is too short. Minimum allowed deadline is 10s.


> Gemini 3.x 모델은 thinking 때문에 짧은 질문도 응답에 수 초가 걸린다(테스트에서 약 8초). 생성 요청의 timeout은 **최소 10초, 넉넉하게 60초 이상**으로 잡는 것이 안전하다. 응답이 길다면 스트리밍(`generate_content_stream`)을 고려한다.

## 4. 연결 오류 (서버에 닿지 못함)

아무 서버도 없는 주소로 요청한다. 실제 키가 엉뚱한 곳으로 전송되지 않도록 가짜 키를 쓴다.

In [6]:
offline_client = genai.Client(
    api_key="dummy",
    http_options=types.HttpOptions(
        base_url="http://127.0.0.1:9",  # 아무 서버도 없는 주소
        retry_options=types.HttpRetryOptions(attempts=1),
    ),
)

try:
    offline_client.models.count_tokens(model=MODEL, contents="안녕")
except httpx.ConnectError as e:
    print("예외 클래스 :", f"{type(e).__module__}.{type(e).__name__}")
    print("메시지      :", e)

예외 클래스 : httpx.ConnectError
메시지      : [Errno 111] Connection refused


## 5. 요청 내용 오류: 404, 400

- 404: `count_tokens`(무료)로 모델 이름 오타를 재현한다.
- 400: 범위를 벗어난 `temperature`(허용 범위 0.0~2.0)로 재현한다. 서버가 검사 단계에서 거절하므로 요금이 발생하지 않는다.

In [7]:
# 404: 모델 이름 오타
try:
    client.models.count_tokens(model="gemini-3.8-flashh", contents="안녕")
except errors.ClientError as e:
    print(f"[{e.code} {e.status}] {e.message[:90]}...")

# 400: 파라미터 범위 오류
try:
    client.models.generate_content(
        model=MODEL, contents="안녕",
        config=types.GenerateContentConfig(temperature=5),
    )
except errors.ClientError as e:
    print(f"[{e.code} {e.status}] {e.message.strip()}")

[404 NOT_FOUND] models/gemini-3.8-flashh is not found for API version v1beta, or is not supported for coun...
[400 INVALID_ARGUMENT] * GenerateContentRequest.generation_config.temperature: temperature must be in the range [0.0, 2.0].


> `count_tokens`는 모든 규칙을 검사하지 않는다. 테스트해 보니 **빈 문자열 입력도 `count_tokens`에서는 통과**했다. `count_tokens`가 통과했다고 실제 요청이 반드시 성공하는 것은 아니다. (Claude와 같음)

## 6. Rate Limit (429)

### 확인 방법
- Gemini는 응답 헤더로 남은 한도를 알려주지 않는다. **[AI Studio의 Rate limit 페이지](https://aistudio.google.com/rate-limit)** 에서 모델별 한도(RPM, TPM, RPD)와 사용률을 확인한다.
- 한도는 **프로젝트 단위**이고, **RPD(일일 요청 수)는 미국 태평양 시간 자정**에 초기화된다. (01 노트북에서 오늘 요청 수를 세는 코드 참고)
- 무료 등급은 한도가 낮고, 결제를 연결하면(Tier 1 이상) 한도가 올라간다.
- 실습 계정에서 429를 일부러 일으키려면 요청을 많이 보내야 하므로 여기서는 재현하지 않는다.

### 처리 방법: 재시도 설정

Gemini SDK는 기본으로 재시도하지 않으므로, **`retry_options`를 지정해 재시도를 켜는 것**이 가장 간단하다.

In [8]:
retry_client = genai.Client(
    http_options=types.HttpOptions(
        timeout=120_000,  # 2분
        retry_options=types.HttpRetryOptions(
            attempts=5,           # 최초 1번 + 재시도 4번
            initial_delay=1.0,    # 첫 재시도 전 1초 대기, 이후 2배씩 증가 (1, 2, 4, 8초...)
            max_delay=60.0,       # 대기 시간 최대 60초
            # http_status_codes를 지정하지 않으면 408, 429, 500, 502, 503, 504를 재시도
        ),
    )
)
print("재시도 설정된 클라이언트 생성 완료")

재시도 설정된 클라이언트 생성 완료


재시도까지 실패한 429는 이렇게 처리한다. 응답 상세 정보에 재시도 대기 시간(`RetryInfo`의 `retryDelay`)이 들어 있으면 그 값을 쓴다.

```python
def retry_delay(e: errors.APIError) -> str | None:
    """429 응답에 서버가 알려준 대기 시간이 있으면 반환 (예: '30s')"""
    try:
        for d in e.details["error"].get("details", []):
            if d.get("@type", "").endswith("RetryInfo"):
                return d.get("retryDelay")
    except (TypeError, KeyError, AttributeError):
        pass
    return None

try:
    response = retry_client.models.generate_content(model=MODEL, contents="...")
except errors.ClientError as e:
    if e.code == 429:
        print(f"요청 한도 초과. {retry_delay(e) or '잠시'} 후 다시 시도하세요.")
    else:
        raise
```

> 429는 분당 한도 초과뿐 아니라 **일일 한도 초과, 지출 한도 초과**일 때도 온다. 일일 한도라면 재시도해도 태평양 시간 자정까지 계속 실패하므로, 메시지를 확인해 원인을 구분한다.

## 7. 종합: 실제로 쓸 수 있는 형태

Claude 05에서 이야기한 원칙과 같다. **대응이 달라지는 만큼만 나눈다.**

- 일시적 문제(429, 5xx, 시간 초과, 연결 오류) → SDK 재시도에 맡기고, 그래도 실패하면 "나중에 다시"
- 설정 문제(잘못된 키, 402 크레딧 소진, 403) → 시작할 때 확인하고 중단
- 코드 버그(400 파라미터 오류, 404 모델 오타) → 잡지 말고 드러나게

In [9]:
TRANSIENT_CODES = {408, 429, 500, 502, 503, 504}


def ask(prompt: str) -> str | None:
    """일시적 문제만 처리하고, 나머지 에러는 그대로 드러나게 둠"""
    try:
        response = retry_client.models.generate_content(model=MODEL, contents=prompt)
    except errors.APIError as e:
        if e.code in TRANSIENT_CODES:
            print(f"[일시적 실패] {e.code} {e.status} → 재시도까지 실패. 나중에 다시 시도")
            return None
        raise  # 400, 402, 403, 404 등은 고쳐야 할 문제이므로 다시 발생시킴
    except (httpx.TimeoutException, httpx.ConnectError) as e:
        print(f"[일시적 실패] 네트워크 문제 ({type(e).__name__}) → 나중에 다시 시도")
        return None
    return response.text


# 무료로 동작을 확인하기 위해, 실패하는 요청만 시험해 본다
for name, fn in [
    ("잘못된 키", lambda: bad_client.models.list(config={"page_size": 1})),
    ("모델 오타", lambda: client.models.count_tokens(model="gemini-3.8-flashh", contents="안녕")),
]:
    try:
        fn()
    except errors.ClientError as e:
        kind = "일시적" if e.code in TRANSIENT_CODES else "고쳐야 할 문제"
        print(f"{name:<6} → {e.code} {e.status} ({kind})")

잘못된 키  → 400 INVALID_ARGUMENT (고쳐야 할 문제)
모델 오타  → 404 NOT_FOUND (고쳐야 할 문제)


## 정리

| 확인하고 싶은 것 | 방법 |
|---|---|
| 연결과 키 | `models.list(config={"page_size": 1})` (무료) |
| 키가 잘못됐는지 | `ClientError` + `reason == "API_KEY_INVALID"` (코드는 400) |
| 요청 기본 형식 | `count_tokens` (무료). 모든 규칙을 검사하지는 않음 |
| 남은 Rate Limit | [AI Studio Rate limit 페이지](https://aistudio.google.com/rate-limit) (헤더 없음) |
| 재시도와 timeout | **직접 설정 필수**: `HttpOptions(timeout=..., retry_options=HttpRetryOptions(...))` |
| 서비스 장애 여부 | [AI Studio Status](https://aistudio.google.com/status) |